# Clustering with k-means
**Machine Learning · Week 6 (Learning without labels) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/ml/w6d1.html

No labels at all: just points. **k-means** finds $k$ groups by repeating two steps:
assign each point to its nearest center, then move each center to the mean of its points.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons

X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=7)   # we throw the labels away

def kmeans(X, k, steps=20, seed=0, centers=None):
    rng = np.random.default_rng(seed)
    if centers is None:
        centers = X[rng.choice(len(X), k, replace=False)]                   # start at k random points
    for _ in range(steps):
        distances = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)   # (points, k)
        assign = distances.argmin(axis=1)                                   # step 1: nearest center
        centers = np.array([X[assign == j].mean(axis=0) if np.any(assign == j) else centers[j]
                            for j in range(k)])                             # step 2: move to the mean
    inertia = ((X - centers[assign]) ** 2).sum()
    return assign, centers, inertia

assign, centers, inertia = kmeans(X, k=4)
palette = np.array(["#7b5cfa", "#17a98c", "#ff8a50", "#5aa9f2", "#e45fa3", "#f0ac2e"])
plt.scatter(X[:, 0], X[:, 1], c=palette[assign], s=14)
plt.scatter(centers[:, 0], centers[:, 1], marker="*", s=420, c="#ffd66b", edgecolors="#2c2842", linewidths=1.5)
plt.title(f"k-means from scratch, k = 4 (inertia {inertia:.0f})")
plt.show()

## How many clusters? The elbow and the silhouette
**Inertia** always drops as $k$ grows, but sharply only until $k$ reaches the real number of groups:
look for the elbow. The **silhouette score** compares, for every point, the mean distance to its own
cluster ($a$) with the mean distance to the nearest other cluster ($b$): $s = \frac{b - a}{\max(a, b)}$,
averaged over points. Close to 1: tight, well-separated clusters. The best $k$ has the highest score.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks = range(1, 10)
models = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X) for k in ks]
silhouettes = [silhouette_score(X, m.labels_) for m in models[1:]]          # needs at least 2 clusters

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2), layout="constrained")
ax1.plot(ks, [m.inertia_ for m in models], "o-", color="#7b5cfa")
ax1.set_xlabel("number of clusters k")
ax1.set_ylabel("inertia")
ax1.set_title("The elbow is at k = 4")
ax2.plot(list(ks)[1:], silhouettes, "o-", color="#17a98c")
ax2.set_xlabel("number of clusters k")
ax2.set_ylabel("silhouette score")
ax2.set_title(f"Highest at k = {int(np.argmax(silhouettes)) + 2}")
plt.show()

## When k-means gets it wrong
k-means draws straight borders halfway between centers, so it expects round groups of similar size.
Long stripes and curved moons break that assumption. A density-based method, DBSCAN, follows the moons.

In [ ]:
from sklearn.cluster import DBSCAN

Xs, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.6, random_state=170)
Xs = Xs @ np.array([[0.6, -0.6], [-0.4, 0.8]])                            # stretch the blobs into stripes
Xm, _ = make_moons(n_samples=300, noise=0.06, random_state=0)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.1), layout="constrained")
for ax, data, k, title in [(axes[0], Xs, 3, "k-means on stripes"), (axes[1], Xm, 2, "k-means on moons")]:
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(data)
    ax.scatter(data[:, 0], data[:, 1], c=palette[labels], s=10)
    ax.set_title(title)
labels = DBSCAN(eps=0.2, min_samples=5).fit_predict(Xm)
axes[2].scatter(Xm[:, 0], Xm[:, 1], c=palette[labels % len(palette)], s=10)
axes[2].set_title("DBSCAN on moons")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## A glimpse of hierarchical clustering
Start with every point as its own cluster and keep merging the two closest clusters.
The **dendrogram** records the merges; cutting it at some height gives the clusters.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

merges = linkage(X[:40], method="ward")
plt.figure(figsize=(8, 3.4))
dendrogram(merges, color_threshold=8, above_threshold_color="#8a85a3", no_labels=True)
plt.axhline(8, color="#ff8a50", ls="--", lw=1.5)
plt.ylabel("merge distance")
plt.title("Dendrogram of 40 points (cutting at the dashed line gives 4 clusters)")
plt.show()

## Your turn: k-means++ by hand
Random starts can drop two centers into the same blob. **k-means++** picks the first center at random,
then each next one at random **with probability proportional to its squared distance** to the nearest
center chosen so far, so far-away points are likely picks. Write it, then compare the final inertia of
20 runs with random starts and 20 runs with k-means++ starts (on `X`, with $k = 4$).

In [ ]:
def kmeans_pp_init(X, k, rng):
    centers = [X[rng.integers(len(X))]]
    while len(centers) < k:
        # TODO: d2 = squared distance from every point to its nearest chosen center
        # TODO: pick the next center with probabilities d2 / d2.sum()   (rng.choice(len(X), p=...))
        centers.append(X[rng.integers(len(X))])     # placeholder: a uniformly random point
    return np.array(centers)

random_runs = [kmeans(X, 4, seed=s)[2] for s in range(20)]
pp_runs = [kmeans(X, 4, centers=kmeans_pp_init(X, 4, np.random.default_rng(s)))[2] for s in range(20)]
print(f"random starts : mean inertia {np.mean(random_runs):.0f}, worst {np.max(random_runs):.0f}")
print(f"k-means++     : mean inertia {np.mean(pp_runs):.0f}, worst {np.max(pp_runs):.0f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def kmeans_pp_init(X, k, rng):
    centers = [X[rng.integers(len(X))]]
    while len(centers) < k:
        d2 = ((X[:, None, :] - np.array(centers)[None, :, :]) ** 2).sum(axis=2).min(axis=1)
        centers.append(X[rng.choice(len(X), p=d2 / d2.sum())])
    return np.array(centers)

random_runs = [kmeans(X, 4, seed=s)[2] for s in range(20)]
pp_runs = [kmeans(X, 4, centers=kmeans_pp_init(X, 4, np.random.default_rng(s)))[2] for s in range(20)]
print(f"random starts : mean inertia {np.mean(random_runs):.0f}, worst {np.max(random_runs):.0f}, "
      f"stuck in {sum(r > 1.1 * min(random_runs) for r in random_runs)} of 20 runs")
print(f"k-means++     : mean inertia {np.mean(pp_runs):.0f}, worst {np.max(pp_runs):.0f}, "
      f"stuck in {sum(r > 1.1 * min(random_runs) for r in pp_runs)} of 20 runs")
print("k-means++ makes bad starts much rarer, not impossible. That's why scikit-learn also restarts several")
print("times (n_init) and keeps the run with the lowest inertia.")